<a href="https://colab.research.google.com/github/JuanEntrena18/proyecto_ansible/blob/main/Guia_despliegue_0_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 Ansible Visual - Guía de Despliegue (Fase 1)

**Objetivo:** Crear un servidor que aloje un editor visual (estilo Unreal Engine) capaz de escanear la red local, detectar sistemas operativos y generar nodos automáticamente.

**Requisitos:**

  * Servidor Ubuntu 22.04 LTS (Máquina Virtual o Física).
  * Conexión a internet.
  * **Importante:** Si usas VirtualBox, configura la red en **"Adaptador Puente" (Bridged)** para poder escanear tu red local real.

-----

## 1\. Preparación del Sistema y Dependencias

Instalaremos Nginx (Web Server), Python (Backend), Ansible (Motor) y Nmap (Escáner).

```bash
# 1. Actualizar repositorios
sudo apt update && sudo apt upgrade -y

# 2. Instalar herramientas básicas
sudo apt install -y software-properties-common curl git ufw

# 3. Instalar Ansible (Motor de IaC)
sudo add-apt-repository --yes --update ppa:ansible/ansible
sudo apt install -y ansible-core

# 4. Instalar Nginx (Servidor Web y Proxy)
sudo apt install -y nginx

# 5. Instalar Python venv y Nmap (Escáner de red)
sudo apt install -y python3-venv nmap
```

-----

## 2\. Configuración del Backend (API Python)

El "cerebro" del sistema. Usaremos FastAPI para ejecutar comandos de sistema.

### 2.1. Crear directorios y entorno virtual

```bash
# 1. Crear carpeta de la API (cambia 'tu_usuario' por tu usuario real, ej: juan)
sudo mkdir -p /opt/ansible-visual/api
sudo chown -R $USER:$USER /opt/ansible-visual

# 2. Crear entorno virtual aislado
cd /opt/ansible-visual/api
python3 -m venv venv

# 3. Activar entorno e instalar librerías
source venv/bin/activate
pip install fastapi uvicorn gunicorn
deactivate
```

### 2.2. Código del Backend (`main.py`)

Crea el archivo de la aplicación:

```bash
nano /opt/ansible-visual/api/main.py
```

**Pega el siguiente código:**

In [ ]:
from fastapi import FastAPI
import subprocess
import re

app = FastAPI()

@app.get("/")
def read_root():
    return {"estado": "OK", "mensaje": "Backend Ansible Visual Activo"}

@app.get("/scan")
def scan_network(subnet: str = "192.168.1.0/24"):
    try:
        # Validación básica de seguridad
        if not re.match(r"^[0-9./]+$", subnet):
             return {"estado": "ERROR", "detalle": "Formato de red inválido"}

        # Ejecutamos Nmap con detección de versiones (-sV) y sin ping (-Pn)
        # Usamos ruta absoluta /usr/bin/nmap por seguridad
        comando = ["/usr/bin/nmap", "-p", "22", "-sV", "-Pn", "--open", "-oG", "-", subnet]

        resultado = subprocess.run(comando, capture_output=True, text=True)

        equipos_encontrados = []

        for linea in resultado.stdout.splitlines():
            # Buscamos líneas de host con puerto 22 abierto
            if "Host:" in linea and "22/open" in linea:
                # 1. Extraer IP
                ip_match = re.search(r'Host: ([0-9.]+)', linea)
                if ip_match:
                    ip = ip_match.group(1)

                    # 2. Detectar S.O. basado en el banner SSH
                    os_detectado = "Linux Genérico"
                    icon_os = "fas fa-server"

                    version_match = re.search(r'//ssh//(.*?)/', linea)
                    if version_match:
                        banner = version_match.group(1).lower()
                        if "ubuntu" in banner:
                            os_detectado = "Ubuntu Server"
                            icon_os = "fab fa-ubuntu"
                        elif "debian" in banner:
                            os_detectado = "Debian"
                            icon_os = "fab fa-linux"
                        elif "raspbian" in banner:
                            os_detectado = "Raspberry Pi"
                            icon_os = "fab fa-raspberry-pi"
                        elif "windows" in banner:
                            os_detectado = "Windows (SSH)"
                            icon_os = "fab fa-windows"

                    equipos_encontrados.append({
                        "ip": ip,
                        "os": os_detectado,
                        "icon": icon_os
                    })

        return {"estado": "OK", "equipos": equipos_encontrados}

    except Exception as e:
        print(f"Error interno: {e}")
        return {"estado": "ERROR", "detalle": str(e)}

### 2.3. Configurar Servicio Systemd

Para que la API se ejecute siempre en segundo plano.

```bash
sudo nano /etc/systemd/system/ansible-api.service
```

**Pega esto (IMPORTANTE: Cambia `User=juan` y `Group=juan` por tu usuario real):**

```ini
[Unit]
Description=Gunicorn instance to serve Ansible Visual API
After=network.target

[Service]
# --- CAMBIA ESTO POR TU USUARIO ---
User=juan
Group=juan
# ----------------------------------

WorkingDirectory=/opt/ansible-visual/api
Environment="PATH=/opt/ansible-visual/api/venv/bin"
ExecStart=/opt/ansible-visual/api/venv/bin/gunicorn -w 1 -k uvicorn.workers.UvicornWorker -b 127.0.0.1:8000 main:app

[Install]
WantedBy=multi-user.target
```

**Activar el servicio:**

```bash
sudo systemctl daemon-reload
sudo systemctl start ansible-api
sudo systemctl enable ansible-api
```

-----

## 3\. Configuración del Frontend (Nginx + HTML)

### 3.1. Configurar Nginx (Reverse Proxy)

```bash
sudo nano /etc/nginx/sites-available/default
```

**Modifica el archivo para que quede así:**

```nginx
server {
    listen 80 default_server;
    server_name _;

    # Ruta donde estará nuestro HTML
    root /var/www/ansible-visual/html;
    index index.html;

    location / {
        try_files $uri $uri/ /index.html;
    }

    # Proxy reverso para conectar con Python
    location /api/ {
        proxy_pass http://127.0.0.1:8000/;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
    }
}
```

**Aplicar cambios:**

```bash
sudo systemctl restart nginx
```

### 3.2. Desplegar el Código Visual (`index.html`)

```bash
# 1. Crear directorio
sudo mkdir -p /var/www/ansible-visual/html
sudo chown -R $USER:$USER /var/www/ansible-visual

# 2. Crear archivo HTML
nano /var/www/ansible-visual/html/index.html
```

**Pega el código completo del Frontend (AdminLTE + Drawflow + Anime.js):**

```html
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Ansible Visual | Auto-Discovery</title>
  <link rel="stylesheet" href="https://fonts.googleapis.com/css?family=Source+Sans+Pro:300,400,400i,700&display=fallback">
  <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
  <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/admin-lte@3.2/dist/css/adminlte.min.css">
  <link rel="stylesheet" href="https://cdn.jsdelivr.net/gh/jerosoler/Drawflow/dist/drawflow.min.css">
  <link rel="stylesheet" href="https://cdn.jsdelivr.net/gh/jerosoler/Drawflow/dist/drawflow.theme.css">
  <style>
    #drawflow { width: 100%; height: calc(100vh - 57px); background: #2b2b2b; background-image: radial-gradient(#444 1px, transparent 1px); background-size: 25px 25px; position: relative; overflow: hidden; outline: none; }
    .drawflow .drawflow-node { background: #1e1e1e; border: 1px solid #444; color: white; width: 250px; border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.5); padding: 0; }
    .drawflow .drawflow-node .title-box { background: #3f6ad8; color: white; padding: 8px 12px; border-radius: 8px 8px 0 0; font-weight: bold; display: flex; align-items: center; gap: 8px; }
    .drawflow .drawflow-node.node-scanner .title-box { background: #6f42c1; }
    .drawflow .drawflow-node.node-machine .title-box { background: #28a745; }
    .drawflow .drawflow-node .box { padding: 10px; }
    .node-input { background: #333; border: 1px solid #555; color: #fff; width: 100%; padding: 5px; border-radius: 4px; margin-bottom: 5px; }
  </style>
</head>
<body class="hold-transition sidebar-mini layout-fixed layout-navbar-fixed dark-mode">
<div class="wrapper">
  <nav class="main-header navbar navbar-expand navbar-dark">
    <ul class="navbar-nav">
      <li class="nav-item"><a class="nav-link" data-widget="pushmenu" href="#"><i class="fas fa-bars"></i></a></li>
      <li class="nav-item d-none d-sm-inline-block"><span class="nav-link font-weight-bold">Ansible Visual Editor</span></li>
    </ul>
  </nav>
  <aside class="main-sidebar sidebar-dark-primary elevation-4">
    <a href="#" class="brand-link"><span class="brand-text font-weight-light px-3">Ansible <b>Visual</b></span></a>
    <div class="sidebar">
      <nav class="mt-2">
        <ul class="nav nav-pills nav-sidebar flex-column" data-widget="treeview" role="menu">
          <li class="nav-header">RED</li>
          <li class="nav-item"><a href="#" class="nav-link" onclick="addNodeToBoard('scanner')"><i class="nav-icon fas fa-satellite-dish text-purple"></i><p>Scanner Red</p></a></li>
          <li class="nav-header">INFRAESTRUCTURA</li>
          <li class="nav-item"><a href="#" class="nav-link" onclick="addNodeToBoard('playbook')"><i class="nav-icon fas fa-file-code text-danger"></i><p>Playbook</p></a></li>
        </ul>
      </nav>
    </div>
  </aside>
  <div class="content-wrapper"><div id="drawflow"></div></div>
</div>
<script src="https://code.jquery.com/jquery-3.6.0.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/bootstrap@4.6.2/dist/js/bootstrap.bundle.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/admin-lte@3.2/dist/js/adminlte.min.js"></script>
<script src="https://cdn.jsdelivr.net/gh/jerosoler/Drawflow/dist/drawflow.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/animejs/3.2.1/anime.min.js"></script>
<script>
  var id = document.getElementById("drawflow");
  const editor = new Drawflow(id); editor.reroute = true; editor.start();

  function addNodeToBoard(tipo, paramPosX, paramPosY, datosExtra) {
    var posX = paramPosX || 100 + (Math.random() * 50);
    var posY = paramPosY || 100 + (Math.random() * 50);
    var nodeId = null;
    if(tipo === 'scanner') {
       var html = `<div class="title-box"><i class="fas fa-satellite-dish"></i> Escáner</div><div class="box"><small>Rango CIDR</small><input type="text" class="node-input subnet-input" value="192.168.1.0/24"><button class="btn btn-block btn-outline-light btn-sm mt-2" onclick="ejecutarEscaneo(this)"><i class="fas fa-radar"></i> Escanear y Mapear</button><div class="mt-2"><small class="text-muted status-text">Listo.</small></div></div>`;
       nodeId = editor.addNode('scanner', 0, 1, posX, posY, 'node-scanner', {}, html, false);
    } else if(tipo === 'machine') {
       var ip = datosExtra.ip || '0.0.0.0'; var os = datosExtra.os || 'Linux'; var icon = datosExtra.icon || 'fas fa-server';
       var html = `<div class="title-box"><i class="${icon}"></i> ${os}</div><div class="box"><div class="d-flex align-items-center mb-2"><i class="fas fa-network-wired mr-2 text-success"></i><input type="text" class="node-input mb-0" value="${ip}" readonly style="font-family:monospace; font-weight:bold;"></div><small class="text-muted"><i class="fas fa-user-shield"></i> Usuario SSH</small><input type="text" class="node-input" placeholder="ej: ubuntu"></div>`;
       nodeId = editor.addNode('machine', 1, 0, posX, posY, 'node-machine', {ip: ip}, html, false);
    } else if(tipo === 'playbook') {
       var html = `<div class="title-box"><i class="fas fa-play-circle"></i> Playbook</div><div class="box"><small>Nombre</small><input type="text" class="node-input" value="site.yml"></div>`;
       nodeId = editor.addNode('playbook', 0, 1, posX, posY, 'node-playbook', {}, html, false);
    }
    if(nodeId) { var nodeElement = document.getElementById('node-' + nodeId); anime({ targets: nodeElement, scale: [0, 1], opacity: [0, 1], duration: 800, easing: 'easeOutElastic(1, .8)' }); }
    return nodeId;
  }

  function ejecutarEscaneo(btn) {
      var $btn = $(btn); var $box = $btn.closest('.box'); var $status = $box.find('.status-text');
      var scannerNodeId = $btn.closest('.drawflow-node').attr('id').replace('node-', '');
      var subnet = $box.find('.subnet-input').val();
      var originalText = $btn.html();
      $btn.html('<i class="fas fa-spinner fa-spin"></i> Escaneando...').prop('disabled', true);
      $status.html('Analizando red...');
      $.ajax({
          url: '/api/scan?subnet=' + subnet, method: 'GET',
          success: function(res) {
              if(res.equipos && res.equipos.length > 0) {
                  $status.html('<span class="text-success">¡' + res.equipos.length + ' encontrados!</span>');
                  var scannerNode = editor.getNodeFromId(scannerNodeId);
                  var startX = scannerNode.pos_x + 350; var startY = scannerNode.pos_y - ((res.equipos.length * 150) / 2);
                  res.equipos.forEach((equipo, index) => {
                      var machineId = addNodeToBoard('machine', startX, startY + (index * 160), {ip: equipo.ip, os: equipo.os, icon: equipo.icon});
                      if(machineId) editor.addConnection(scannerNodeId, machineId, "output_1", "input_1");
                  });
              } else { $status.html('<span class="text-danger">Sin resultados.</span>'); }
          },
          error: function() { $status.text('Error API.'); },
          complete: function() { $btn.html(originalText).prop('disabled', false); }
      });
  }
  addNodeToBoard('scanner', 100, 200);
</script>
</body>
</html>
```

-----

## 4\. Firewall y Seguridad Final

Abre los puertos para que puedas entrar desde tu navegador:

```bash
sudo ufw allow 22/tcp    # SSH
sudo ufw allow 'Nginx Full' # HTTP/HTTPS
sudo ufw enable
```

-----

## 5\. Cómo probarlo

1.  Abre tu navegador en cualquier PC de la red.
2.  Escribe: `http://[IP_DEL_SERVIDOR]/`
3.  Verás el editor oscuro. Escribe tu red (ej: `192.168.1.0/24`) en el nodo Escáner.
4.  Haz clic en **"Escanear y Mapear"**.
5.  El sistema creará automáticamente nodos conectados para cada PC que encuentre con el puerto SSH abierto.